In [1]:
import pandas as pd
import numpy as np
import os
import re
import gc
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from scipy import stats
import time

AA_GENO  = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7_snp_encoded_012.csv"
AA_META  = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
OUT_DIR  = r"C:\Users\user\Desktop\ai causal\causal_project\african_american\smoking_status"

os.makedirs(OUT_DIR, exist_ok=True)
print("Ready. OUT_DIR:", OUT_DIR)

Ready. OUT_DIR: C:\Users\user\Desktop\ai causal\causal_project\african_american\smoking_status


In [2]:
def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# manifest
manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_cores = set(
    manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "core_name"]
)

# metadata
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)

# genotype
encoded_df = pd.read_csv(AA_GENO)
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_ids = encoded_df.columns[1:].tolist()

# align metadata to genotype column order
meta_aligned = meta_df.set_index("sample_id").reindex(sample_ids).reset_index()
Y = (meta_aligned["smoking_status"] == "Smoker").astype(np.float64).values
age_std = ((meta_aligned["age"].astype(float) - meta_aligned["age"].astype(float).mean()) /
           meta_aligned["age"].astype(float).std()).values
gender_binary = (meta_aligned["gender"] == "Male").astype(np.float64).values

print("Samples:", len(sample_ids))
print("Y distribution:", np.unique(Y, return_counts=True))

# autosomal SNPs only
keep_mask = ~np.array([strip_suffix(p) in sex_linked_cores for p in probe_id_array])
X_snp = encoded_df[sample_ids].to_numpy(dtype=np.int8)[keep_mask]
X_auto = X_snp.T
probe_ids_auto = probe_id_array[keep_mask]

del encoded_df, X_snp
gc.collect()
print("X_auto shape:", X_auto.shape)

Samples: 3348
Y distribution: (array([0., 1.]), array([1717, 1631]))
X_auto shape: (3348, 233610)


In [3]:
X_float = X_auto.astype(np.float32)
del X_auto
gc.collect()

p = X_float.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_mask = denom > 1e-8
print("Informative SNPs:", valid_mask.sum())

X_std = (X_float[:, valid_mask] - 2 * p[valid_mask]) / denom[valid_mask]
probe_ids_valid = probe_ids_auto[valid_mask]
del X_float
gc.collect()

pca = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs = pca.fit_transform(X_std)
print("Explained variance:", pca.explained_variance_ratio_)

X_conf = np.hstack([
    pcs,
    age_std.reshape(-1, 1),
    gender_binary.reshape(-1, 1)
]).astype(np.float64)

print("X_conf shape:", X_conf.shape)

Informative SNPs: 141645
Explained variance: [0.00563623 0.00111008 0.00105167 0.00098916 0.00098035 0.00095548
 0.00091716 0.00086775 0.00085743 0.00084649]
X_conf shape: (3348, 12)


In [4]:
def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))

    return theta, pvals

print("Single test iteration...")
start = time.time()
_, pvals_test = doubleml_scan(X_std, Y, X_conf, random_state=0)
print(f"Done in {time.time()-start:.1f}s")
for thresh in [0.01, 0.001, 0.0001]:
    print(f"p < {thresh}: {(pvals_test < thresh).sum()} SNPs")

Single test iteration...
Done in 62.7s
p < 0.01: 1257 SNPs
p < 0.001: 117 SNPs
p < 0.0001: 13 SNPs


In [5]:
import pandas as pd
import os

# reload previous stability results
prev_stability = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\v2_stability_results.csv")

print("Stability distribution:")
for thresh in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    n = (prev_stability["stability_fraction"] >= thresh).sum()
    print(f"  >= {thresh:.0%}: {n} SNPs")

Stability distribution:
  >= 50%: 116 SNPs
  >= 60%: 111 SNPs
  >= 70%: 108 SNPs
  >= 80%: 100 SNPs
  >= 90%: 92 SNPs
  >= 100%: 67 SNPs


In [6]:
import pandas as pd
import numpy as np
import re
import os
import json

OUT_DIR = r"C:\Users\user\Desktop\ai causal\causal_project\african_american\smoking_status"
AA_GENO = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7_snp_encoded_012.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}

# get shortlist at 80% stability
prev_stability = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\v2_stability_results.csv")
shortlist = prev_stability[prev_stability["stability_fraction"] >= 0.80].copy()
shortlist["core_name"] = shortlist["probe_id"].map(strip_suffix)

pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist = shortlist.merge(pos_lookup, on="core_name", how="left")
shortlist = shortlist[~shortlist["Chr"].isin(non_autosomal)].copy()
print("Shortlist at >=80% stability:", len(shortlist))

# load genotype vectors for LD pruning
encoded_df = pd.read_csv(AA_GENO)
sample_ids = encoded_df.columns[1:].tolist()
probe_rows = encoded_df[encoded_df["probe_id"].isin(set(shortlist["probe_id"]))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(shortlist["probe_id"].tolist())
X_shortlist = probe_rows[sample_ids].to_numpy(dtype=np.float64).T
probe_id_to_idx = {pid: i for i, pid in enumerate(shortlist["probe_id"].tolist())}
del encoded_df, probe_rows

def get_geno(pid):
    return X_shortlist[:, probe_id_to_idx[pid]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_geno(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_geno(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained = greedy_ld_prune(shortlist)
shortlist_pruned = shortlist[
    shortlist["probe_id"].isin(retained)
].copy().sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(f"After LD pruning: {len(shortlist_pruned)} SNPs")

# check perfect correlations
encoded_df2 = pd.read_csv(AA_GENO)
sample_ids2 = encoded_df2.columns[1:].tolist()
probe_rows2 = encoded_df2[encoded_df2["probe_id"].isin(set(shortlist_pruned["probe_id"]))].copy()
probe_rows2 = probe_rows2.set_index("probe_id").reindex(shortlist_pruned["probe_id"].tolist())
X_check = probe_rows2[sample_ids2].to_numpy(dtype=np.float64).T
corr_matrix = np.corrcoef(X_check.T)

to_remove = set()
for i in range(len(shortlist_pruned)):
    for j in range(i+1, len(shortlist_pruned)):
        if abs(corr_matrix[i,j]) > 0.99 and shortlist_pruned["probe_id"].iloc[j] not in to_remove:
            to_remove.add(shortlist_pruned["probe_id"].iloc[j])

print(f"Perfectly correlated to remove: {len(to_remove)}")
shortlist_final = shortlist_pruned[~shortlist_pruned["probe_id"].isin(to_remove)].copy()
print(f"Final shortlist: {len(shortlist_final)} SNPs")

shortlist_final.to_csv(os.path.join(OUT_DIR, "shortlist_final.csv"), index=False)

# build PC input
AA_META = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv"
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_aligned = meta_df.set_index("sample_id").reindex(sample_ids2).reset_index()
Y = (meta_aligned["smoking_status"] == "Smoker").astype(np.float64).values

final_ids = shortlist_final["probe_id"].tolist()
col_names = final_ids + ["smoking_status"]

probe_rows_final = encoded_df2[encoded_df2["probe_id"].isin(set(final_ids))].copy()
probe_rows_final = probe_rows_final.set_index("probe_id").reindex(final_ids)
X_pc = probe_rows_final[sample_ids2].to_numpy(dtype=np.float64).T
X_pc_full = np.hstack([X_pc, Y.reshape(-1, 1)])

print("PC input shape:", X_pc_full.shape)

np.save(os.path.join(OUT_DIR, "pc_input.npy"), X_pc_full)
with open(os.path.join(OUT_DIR, "pc_col_names.json"), "w") as f:
    json.dump(col_names, f)
print("Saved.")

Shortlist at >=80% stability: 100
After LD pruning: 88 SNPs
Perfectly correlated to remove: 0
Final shortlist: 88 SNPs
PC input shape: (3348, 89)
Saved.
